# Aula 06 - Notebook: Lógica de Predicados e Quantificadores em Redes de Sensores

Neste notebook, implementamos o **Motor de Varredura de Predicados** aplicável à malha de instrumentos do SCADA da Estação de Reabastecimento de Hidrogênio. A modelagem substitui lógicas hardcoded estáticas por quantificadores universais ($\forall$, via operador `FORALL`) e existenciais ($\exists$, via operador `EXISTS`) sobre conjuntos dinâmicos de sensores para validar as condições operacionais e acionar intertravamentos de segurança.

In [1]:
from dataclasses import dataclass
from typing import List, Callable, Any, Optional

def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura para relatórios do SCADA."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

# Definição formal do Tipo de Dado da Malha (Universo de Discurso)
@dataclass
class AtivoIndustrial:
    tag: str
    setor: str
    tipo_medicao: str
    valor_atual: float
    unidade: str
    limite_critico: Optional[float] = None
    estado_discreto: Optional[bool] = None

# Operadores de Lógica de Primeira Ordem (FOL)
def FORALL(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    """Quantificador Universal: Verdadeiro se TODOS os elementos satisfizerem o predicado."""
    return all(predicado(x) for x in dominio)

def EXISTS(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    """Quantificador Existencial: Verdadeiro se AO MENOS UM elemento satisfizer o predicado."""
    return any(predicado(x) for x in dominio)

print("[INFO] Motor de quantificação e modelo de rede de sensores carregados com sucesso.")

[INFO] Motor de quantificação e modelo de rede de sensores carregados com sucesso.


In [2]:
# Instanciando o Universo de Sensores e Atuadores
rede_scada = [
    AtivoIndustrial('PT-101', 'Setor 100', 'PRESSAO', 380.0, 'bar', 400.0),
    AtivoIndustrial('PT-102', 'Setor 100', 'PRESSAO', 680.0, 'bar', 700.0),
    AtivoIndustrial('TT-101', 'Setor 100', 'TEMPERATURA', 42.0, 'ºC', 85.0),
    AtivoIndustrial('AT-101', 'Setor 100', 'GAS_H2', 5.0, '% LIE', 25.0),
    AtivoIndustrial('TT-201', 'Setor 200', 'TEMPERATURA', -42.0, 'ºC', -40.0), # Chiller (deve ser <= limite)
    AtivoIndustrial('M-201', 'Setor 200', 'DISCRETO', 0.0, '', estado_discreto=True),
    AtivoIndustrial('COM-301', 'Setor 300', 'DISCRETO', 0.0, '', estado_discreto=True),
    AtivoIndustrial('BV-301', 'Setor 300', 'DISCRETO', 0.0, '', estado_discreto=True),
    AtivoIndustrial('HS-301', 'Setor 300', 'DISCRETO', 0.0, '', estado_discreto=True),
    AtivoIndustrial('ESD-100', 'Global', 'DISCRETO', 0.0, '', estado_discreto=False) # Parada de Emergencia
]

# 1. Filtrando os subdomínios lógicos
S100 = [s for s in rede_scada if s.setor == 'Setor 100']

# 2. Definição Dinâmica de Predicados
def em_falha_analogica(sensor: AtivoIndustrial) -> bool:
    if sensor.limite_critico is not None:
        # Regra do chiller: temperatura DEVE ser menor que -40
        if sensor.tag == 'TT-201': 
            return sensor.valor_atual > sensor.limite_critico
        # Demais pressões, temps e gás do setor 100
        return sensor.valor_atual > sensor.limite_critico 
    return False

def ativo_acionado(tag: str) -> bool:
    alvo = next((s for s in rede_scada if s.tag == tag), None)
    return alvo.estado_discreto if alvo else False

# 3. Processamento dos Quantificadores
# Trip (∃x ∈ S100: Falha(x) ∨ Emergência)
existe_falha_s100 = EXISTS(S100, em_falha_analogica)
trip_sis = existe_falha_s100 or ativo_acionado('ESD-100')

# Integridade (∀x ∈ S100: ¬Falha(x))
integridade_s100 = FORALL(S100, lambda s: not em_falha_analogica(s))

# Permissivo de Dispensação
sensor_chiller = next((s for s in rede_scada if s.tag == 'TT-201'), None)
permissivo_dispensador = (
    integridade_s100 and 
    not em_falha_analogica(sensor_chiller) and
    ativo_acionado('M-201') and
    ativo_acionado('COM-301') and
    ativo_acionado('BV-301') and
    ativo_acionado('HS-301') and
    not ativo_acionado('ESD-100')
)

print("=== STATUS SCADA: ESTAÇÃO DE HIDROGÊNIO ===\n")
print(f"1. Falha Existencial no Banco (TRIP): {trip_sis}")
print(f"   -> (EXISTS sensor em S100 com valor > limite)\n")
print(f"2. Integridade Universal do Setor 100: {integridade_s100}")
print(f"   -> (FORALL sensores em S100 com valor <= limite)\n")
print(f"3. Permissivo de Abertura do Dispensador (XV-301): {permissivo_dispensador}\n")

print("=== RELATÓRIO DO DOMÍNIO S100 ===")
dados_s100 = [{
    "Tag": s.tag, "Setor": s.setor, "Tipo": s.tipo_medicao, 
    "Valor": f"{s.valor_atual} {s.unidade}", "Limite": f"{s.limite_critico} {s.unidade}", 
    "Alarme": em_falha_analogica(s)
} for s in S100]
print(formatar_tabela(dados_s100))

=== STATUS SCADA: ESTAÇÃO DE HIDROGÊNIO ===

1. Falha Existencial no Banco (TRIP): False
   -> (EXISTS sensor em S100 com valor > limite)

2. Integridade Universal do Setor 100: True
   -> (FORALL sensores em S100 com valor <= limite)

3. Permissivo de Abertura do Dispensador (XV-301): True

=== RELATÓRIO DO DOMÍNIO S100 ===
Tag    | Setor     | Tipo        | Valor     | Limite    | Alarme
-------+-----------+-------------+-----------+-----------+-------
PT-101 | Setor 100 | PRESSAO     | 380.0 bar | 400.0 bar | False 
PT-102 | Setor 100 | PRESSAO     | 680.0 bar | 700.0 bar | False 
TT-101 | Setor 100 | TEMPERATURA | 42.0 ºC   | 85.0 ºC   | False 
AT-101 | Setor 100 | GAS_H2      | 5.0 % LIE | 25.0 % LIE| False 
